# Object Detection — YOLO from Scratch Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: IoU

The workhorse of the whole lesson. Works on two arrays of boxes in `(x1, y1, x2, y2)` format.

In [ ]:
```python

import numpy as np

def box_iou(boxes_a, boxes_b):

    ax1, ay1, ax2, ay2 = boxes_a[:, 0], boxes_a[:, 1], boxes_a[:, 2], boxes_a[:, 3]

    bx1, by1, bx2, by2 = boxes_b[:, 0], boxes_b[:, 1], boxes_b[:, 2], boxes_b[:, 3]

    inter_x1 = np.maximum(ax1[:, None], bx1[None, :])

    inter_y1 = np.maximum(ay1[:, None], by1[None, :])

    inter_x2 = np.minimum(ax2[:, None], bx2[None, :])

    inter_y2 = np.minimum(ay2[:, None], by2[None, :])

    inter_w = np.clip(inter_x2 - inter_x1, 0, None)

    inter_h = np.clip(inter_y2 - inter_y1, 0, None)

    inter = inter_w * inter_h

    area_a = (ax2 - ax1) * (ay2 - ay1)

    area_b = (bx2 - bx1) * (by2 - by1)

    union = area_a[:, None] + area_b[None, :] - inter

    return inter / np.clip(union, 1e-8, None)

In [ ]:
```

Returns an `(N_a, N_b)` matrix of pairwise IoUs. Use it against a single ground-truth box by making one of the arrays shape `(1, 4)`.

### Step 2: Non-max suppression

In [ ]:
```python

def nms(boxes, scores, iou_threshold=0.45):

    order = np.argsort(-scores)

    keep = []

    while len(order) > 0:

        i = order[0]

        keep.append(i)

        if len(order) == 1:

            break

        rest = order[1:]

        ious = box_iou(boxes[[i]], boxes[rest])[0]

        order = rest[ious <= iou_threshold]

    return np.array(keep, dtype=np.int64)

In [ ]:
```

Deterministic, `O(N log N)` from the sort, and matches the behaviour of `torchvision.ops.nms` on identical inputs.

### Step 3: Box encoding and decoding

Convert between pixel coordinates and the `(tx, ty, tw, th)` targets that the network actually regresses.

In [ ]:
```python

def encode(box_xyxy, cell_x, cell_y, stride, anchor_wh):

    x1, y1, x2, y2 = box_xyxy

    cx = 0.5 * (x1 + x2)

    cy = 0.5 * (y1 + y2)

    w = x2 - x1

    h = y2 - y1

    tx = cx / stride - cell_x

    ty = cy / stride - cell_y

    tw = np.log(w / anchor_wh[0] + 1e-8)

    th = np.log(h / anchor_wh[1] + 1e-8)

    return np.array([tx, ty, tw, th])

def decode(tx_ty_tw_th, cell_x, cell_y, stride, anchor_wh):

    tx, ty, tw, th = tx_ty_tw_th

    cx = (sigmoid(tx) + cell_x) * stride

    cy = (sigmoid(ty) + cell_y) * stride

    w = anchor_wh[0] * np.exp(tw)

    h = anchor_wh[1] * np.exp(th)

    return np.array([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])

def sigmoid(x):

    return 1.0 / (1.0 + np.exp(-x))

In [ ]:
```

Test: encode a box then decode — you should get back something very close to the original (up to the sigmoid inverse not being perfectly invertible when `tx` is not in the post-sigmoid range).

### Step 4: A minimal YOLO head

One 1x1 conv on a feature map, reshaping to `(B, S, S, num_anchors, 5 + C)`.

In [ ]:
```python

import torch

import torch.nn as nn

class YOLOHead(nn.Module):

    def __init__(self, in_c, num_anchors, num_classes):

        super().__init__()

        self.num_anchors = num_anchors

        self.num_classes = num_classes

        self.conv = nn.Conv2d(in_c, num_anchors * (5 + num_classes), kernel_size=1)

    def forward(self, x):

        n, _, h, w = x.shape

        y = self.conv(x)

        y = y.view(n, self.num_anchors, 5 + self.num_classes, h, w)

        y = y.permute(0, 3, 4, 1, 2).contiguous()

        return y

In [ ]:
```

Output shape: `(N, H, W, num_anchors, 5 + C)`. The last dimension holds `[tx, ty, tw, th, obj, cls_0, ..., cls_{C-1}]`.

### Step 5: Ground-truth assignment

For every ground-truth box, decide which `(cell, anchor)` is responsible.

In [ ]:
```python

def assign_targets(boxes_xyxy, classes, anchors, stride, grid_size, num_classes):

    num_anchors = len(anchors)

    target = np.zeros((grid_size, grid_size, num_anchors, 5 + num_classes), dtype=np.float32)

    has_obj = np.zeros((grid_size, grid_size, num_anchors), dtype=bool)

    for box, cls in zip(boxes_xyxy, classes):

        x1, y1, x2, y2 = box

        cx, cy = 0.5 * (x1 + x2), 0.5 * (y1 + y2)

        gx, gy = int(cx / stride), int(cy / stride)

        bw, bh = x2 - x1, y2 - y1

        ious = np.array([

            (min(bw, aw) * min(bh, ah)) / (bw * bh + aw * ah - min(bw, aw) * min(bh, ah))

            for aw, ah in anchors

        ])

        best = int(np.argmax(ious))

        aw, ah = anchors[best]

        target[gy, gx, best, 0] = cx / stride - gx

        target[gy, gx, best, 1] = cy / stride - gy

        target[gy, gx, best, 2] = np.log(bw / aw + 1e-8)

        target[gy, gx, best, 3] = np.log(bh / ah + 1e-8)

        target[gy, gx, best, 4] = 1.0

        target[gy, gx, best, 5 + cls] = 1.0

        has_obj[gy, gx, best] = True

    return target, has_obj

In [ ]:
```

Anchor selection is "best shape IoU with the ground truth" — a cheap proxy that matches the YOLOv2/v3 assignment. v5 and later use more sophisticated strategies (task-aligned matching, dynamic k) that refine the same idea.

### Step 6: The three losses

In [ ]:
```python

def yolo_loss(pred, target, has_obj, lambda_coord=5.0, lambda_obj=1.0, lambda_noobj=0.5, lambda_cls=1.0):

    has_obj_t = torch.from_numpy(has_obj).bool()

    target_t = torch.from_numpy(target).float()

    # box-regression loss: only on cells with objects

    box_pred = pred[..., :4][has_obj_t]

    box_true = target_t[..., :4][has_obj_t]

    loss_box = torch.nn.functional.mse_loss(box_pred, box_true, reduction="sum")

    # objectness loss

    obj_pred = pred[..., 4]

    obj_true = target_t[..., 4]

    loss_obj_pos = torch.nn.functional.binary_cross_entropy_with_logits(

        obj_pred[has_obj_t], obj_true[has_obj_t], reduction="sum")

    loss_obj_neg = torch.nn.functional.binary_cross_entropy_with_logits(

        obj_pred[~has_obj_t], obj_true[~has_obj_t], reduction="sum")

    # classification loss on cells with objects

    cls_pred = pred[..., 5:][has_obj_t]

    cls_true = target_t[..., 5:][has_obj_t]

    loss_cls = torch.nn.functional.binary_cross_entropy_with_logits(

        cls_pred, cls_true, reduction="sum")

    total = (lambda_coord * loss_box

             + lambda_obj * loss_obj_pos

             + lambda_noobj * loss_obj_neg

             + lambda_cls * loss_cls)

    return total, {"box": loss_box.item(), "obj_pos": loss_obj_pos.item(),

                   "obj_neg": loss_obj_neg.item(), "cls": loss_cls.item()}

In [ ]:
```

Five hyper-parameters that every YOLO tutorial either hardcodes or sweeps. The ratios matter: `lambda_coord=5, lambda_noobj=0.5` mirrors the original YOLOv1 paper and still works as a reasonable default.

### Step 7: Inference pipeline

Decode the raw head output, apply sigmoid/exp, threshold on objectness, and NMS.

In [ ]:
```python

def postprocess(pred_tensor, anchors, stride, img_size, conf_threshold=0.25, iou_threshold=0.45):

    pred = pred_tensor.detach().cpu().numpy()

    grid_h, grid_w = pred.shape[1], pred.shape[2]

    num_anchors = len(anchors)

    boxes, scores, classes = [], [], []

    for gy in range(grid_h):

        for gx in range(grid_w):

            for a in range(num_anchors):

                tx, ty, tw, th, obj, *cls = pred[0, gy, gx, a]

                score = sigmoid(obj) * sigmoid(np.array(cls)).max()

                if score < conf_threshold:

                    continue

                cls_idx = int(np.argmax(cls))

                cx = (sigmoid(tx) + gx) * stride

                cy = (sigmoid(ty) + gy) * stride

                w = anchors[a][0] * np.exp(tw)

                h = anchors[a][1] * np.exp(th)

                boxes.append([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])

                scores.append(float(score))

                classes.append(cls_idx)

    if not boxes:

        return np.zeros((0, 4)), np.zeros((0,)), np.zeros((0,), dtype=int)

    boxes = np.array(boxes)

    scores = np.array(scores)

    classes = np.array(classes)

    keep = nms(boxes, scores, iou_threshold)

    return boxes[keep], scores[keep], classes[keep]

In [ ]:
```

That is the complete eval path: head -> decode -> threshold -> NMS.

## Exercises

In [ ]:
1. **(Easy)** Implement `box_iou` and run it against `torchvision.ops.box_iou` on 1,000 random box pairs. Verify max absolute difference is below `1e-6`.
2. **(Medium)** Port `yolo_loss` to a version that uses `CIoU` box loss instead of MSE. Show on a 100-image synthetic dataset that CIoU converges to a better final mAP@0.5:0.95 than MSE in the same number of epochs.
3. **(Hard)** Implement multi-scale inference: feed the same image at three resolutions through the model, union the box predictions, and run a single NMS at the end. Measure the mAP lift vs single-scale inference on a held-out set.